# RULE-BASED CELL TYPING WITH BANKSY

This notebook provides the workflow from AnnData object to BANKSY domains, rule-based cell typing, annotation of BANKSY domains as B-cell follicles, and spatially informed cell typing.

Your AnnData object should contain marker positivity information) from the compute_positivity_matrix function in annotation.py).

# BANKSY
For this you need a saved .h5ad file containing an anndata object, and Python 12 is required (not a later version of Python). Note that BANKSY only identifies spatial domains and not cell types.

First, load your H5AD file and choose a "base name" to easily keep track of your file and plot naming.

In [ ]:
basename = "IHOPE39_Spleen" #For example

In [ ]:
from banksy.initialize_banksy import initialize_banksy
from banksy.run_banksy import run_banksy_multiparam
import scanpy as sc

h5ad_file = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"

# Load your spatial transcriptomics data
adata = sc.read_h5ad(h5ad_file)

# Initialize BANKSY
coord_keys = ('x', 'y', 'spatial')
banksy_dict = initialize_banksy(
    adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay='scaled_gaussian'
)

# Run BANKSY clustering

results_df = run_banksy_multiparam(
    adata,
    banksy_dict,
    lambda_list=[0.2],
    resolutions=[0.5, 1.0]
)

In [ ]:
from banksy.main import median_dist_to_nearest_neighbour
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from banksy.plot_banksy import plot_results

# Parameters for BANKSY
coord_keys = ('x', 'y', 'spatial')
k_geom = 15
max_m = 1
nbr_weight_decay = "scaled_gaussian"
lambda_list = [0.8]
resolutions = [0.5]       # Leiden clustering resolution
pca_dims = [20]             # Try a lower number?
cluster_algorithm = 'leiden'
cmap = 'tab20'             # color map for spatial plotting
save_path = None           # e.g., "./BANKSY_results" if you want to save figures

# Compute neighbor distances
nbrs = median_dist_to_nearest_neighbour(adata, key=coord_keys[2])

# Initialize BANKSY
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=False,
    plt_nbr_weights=True,
    plt_agf_angles=False,
    plt_theta=False
)

# Generate BANKSY matrix
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata,
    banksy_dict,
    lambda_list,
    max_m
)

# Dimensionality reduction by UMAP
pca_umap(
    banksy_dict,
    pca_dims=pca_dims,
    add_umap=True
)

# Run Leiden clustering
results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions=resolutions,
    num_nn=50,
    num_iterations=-1,
    partition_seed=1234,
    match_labels=True,
    max_labels=None
)

# Map clusters back to adata
cluster_labels = results_df.labels[results_df.index[0]].dense
adata.obs['banksy_domain'] = cluster_labels.astype(str)

# Optional: visualize
sc.pl.spatial(adata, color='banksy_domain', spot_size=30, title='BANKSY Domains')

# Optional: use BANKSY plotting function
if save_path is not None:
    os.makedirs(save_path, exist_ok=True)
    weights_graph = banksy_dict['scaled_gaussian']['weights'][1]
    plot_results(
        results_df[results_df['num_labels']==len(np.unique(cluster_labels))],
        weights_graph,
        cmap,
        match_labels=True,
        coord_keys=coord_keys,
        max_num_labels=max_num_labels,
        save_path=save_path,
        save_fig=True,
        save_fullfig=True,
        dataset_name='Sample',
        save_labels=True
    )

print(f"BANKSY complete for {basename}! Domains added to `adata.obs['banksy_domain']`")

Save anndata object with BANKSY domains

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
save_h5ad(adata, h5adpath)

# Rule-based cell typing

Each cell type will correspond to a column in the AnnData object, with a boolean True/False for every cell type in every cell.

On the lineage ("type") level, the labels are exclusive.

On the intermediate and subtype levels, labels may overlap.

Set file name/path and basename if needed:

In [ ]:
# Load AnnData
adata = sc.read_h5ad(
    "../data/processed/anndata/IHOPE14_MedLN_BottomLeft_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
)
basename = "IHOPE14_MedLN_BottomLeft"
print(f"Base name: {basename}")

If the AnnData already has cell types, remove them:

In [ ]:
cols_to_drop = [
    c for c in adata.obs.columns
    if c.startswith(("type_", "intermediate_", "subtype_", "state_"))
]

adata.obs.drop(columns=cols_to_drop, inplace=True)

print(f"Removed {len(cols_to_drop)} old annotation columns")

**Cell typing**

In [ ]:
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE

adata = assign_cell_types_bool_IHOPE(adata)

Save:

In [ ]:
from scripts.anndata_helpers import save_h5ad

celltype_path = f"../data/processed/anndata/NEW/{basename}_celltypes.h5ad"

save_h5ad(adata, celltype_path)

# B cell follicle domain annotation

Annotate BANKSY domains, here as B-cell follicles. This analysis is based on B cell enrichment across domains, you can then manually select the domains that are likely to be B cell follicles. You can also choose to set a B cell percentage threshold for what is annotated as a follicle.

In [ ]:
from scripts.banksy_domains import compute_domain_bcell_stats, plot_domains_by_bcell_fraction

stats_df = compute_domain_bcell_stats(adata)
plot_domains_by_bcell_fraction(adata, stats_df, cmap="coolwarm")

In [ ]:
print(f"{basename}_BANKSY_B_cell_percentage_spatial")

Manually select top domains and add to anndata

In [ ]:
top_domains = ['8']  # <-- you set this per sample, within ''

In [ ]:
from scripts.banksy_domains import assign_bcell_follicles

adata = assign_bcell_follicles(
    adata,
    follicle_domains=top_domains,
    banksy_domain_key="banksy_domain",
    output_key="B_follicle",
)

print("Sanity check sum: ", adata.obs["B_follicle"].sum())

Quick visual check of the follicle domain:

In [ ]:
from scripts.banksy_domains import plot_domain_mask
plot_domain_mask(adata, top_domains)

Plot of the B cells within the selected domain


In [ ]:
from scripts.banksy_domains import plot_bcell_follicles

plot_bcell_follicles(adata, sample_name = basename)

In [ ]:
print(f"{basename}_BANKSY_follicles_domain")

Save:

In [ ]:
follicle_path = f"../data/processed/anndata/NEW/{basename}_celltypes_follicledomains.h5ad"

save_h5ad(adata, follicle_path)

# Spatially informed cell typing
Finally, it's time to add the spatially defined cell types.

**TfH-like** inside follicles

In [ ]:
from scripts.celltype_rules_IHOPE import add_TfH_like_cells

adata = add_TfH_like_cells(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

**GC B cells** in follicles and **plasmablasts** outside follicles

In [ ]:
#GC-B AND PLASMABLASTS
from scripts.celltype_rules_IHOPE import add_spatial_B_context

adata = add_spatial_B_context(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

Save:

In [ ]:
final_path = f"../data/processed/anndata/NEW/{basename}_celltypes_follicledomains.h5ad"

save_h5ad(adata, final_path)

# Summarize

Generate a CSV file with cell type counts and percentages on all levels. This can be used in later analysis of the entire dataset.

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv"
)